# Fault sweep — earthing impedance and reduction factors per bus

When the fault location is not known a priori, a common analysis is to
**sweep the fault across every candidate bus** and read off the earthing
impedance $Z_E$ and the reduction factors at each fault site. This notebook
demonstrates that workflow on a 10-bus MV cable line and visualises the
results with `Network.res_all_impedances()`.

Steps:

1. Build a 10-bus MV cable line at five frequencies (50, 250, 350, 450, 550 Hz).
2. Create one fault per bus, plus a single source at `bus01`.
3. Loop `gi.run_fault` over every fault.
4. Read the long-format DataFrame returned by `res_all_impedances()`.
5. Plot $|Z_E|$ and both reduction factors per fault bus and frequency.

In [ ]:
import warnings

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import groundinsight as gi

print("groundinsight", gi.__version__)

## 1. Bus and branch types

Illustrative MV cable values: a shield-earth loop of $0.25 + j0.6\ \Omega$/km
at 50 Hz whose reactance and the mutual reactance both grow with frequency.
The cable has full mutual coupling ($\mathrm{Im}\,Z_\text{self} =
\mathrm{Im}\,Z_\text{mutual}$), so its reduction factor at 50 Hz is
$|1 - Z_\text{mutual}/Z_\text{self}| = 0.25/|0.25 + j0.6| = 0.384615$. Every
station has an electrode of 1 Ω.

In [ ]:
rho = 100.0  # specific earth resistivity in Ohm*m

bus_type = gi.BusType(
    name="BusUnit",
    description="Unit-like bus impedance, dominated by cable shield path",
    system_type="Grounded",
    voltage_level=20.0,
    impedance_formula="rho * 0 + 1.0 + I * f * 0",
)

cable_type = gi.BranchType(
    name="MSCable",
    description="MV cable with shield (constant L, M)",
    grounding_conductor=True,
    self_impedance_formula="(0.25 + I * 0.6 * f / 50) * l",
    mutual_impedance_formula="(0.0  + I * 0.6 * f / 50) * l",
)

## 2. Build a 10-bus line

Source at `bus01`, one fault definition at every bus. Each fault re-uses the
same source; `run_fault` solves one fault at a time.

In [ ]:
FREQS = [50.0, 250.0, 350.0, 450.0, 550.0]
N_BUS = 10
LINE_LENGTH_KM = 0.5  # km per section

net = gi.create_network(name="FaultSweep", frequencies=FREQS,
                        description="10-bus MV cable line, fault sweep demo")

for i in range(1, N_BUS + 1):
    gi.create_bus(name=f"bus{i:02d}", type=bus_type, network=net,
                  specific_earth_resistance=rho)

for i in range(1, N_BUS):
    gi.create_branch(
        name=f"b{i:02d}_{i+1:02d}",
        type=cable_type,
        from_bus=f"bus{i:02d}",
        to_bus=f"bus{i+1:02d}",
        length=LINE_LENGTH_KM,
        specific_earth_resistance=rho,
        network=net,
    )

gi.create_source(name="src", bus="bus01", values={f: 1000.0 for f in FREQS}, network=net)
for i in range(1, N_BUS + 1):
    gi.create_fault(name=f"f{i:02d}", bus=f"bus{i:02d}", network=net)

print(f"{len(net.buses)} buses, {len(net.branches)} branches, "
      f"{len(net.faults)} faults, {len(net.frequencies)} frequencies")

## 3. Run the sweep

One `gi.run_fault` call per fault location. The fault at `bus01` sits on the
source bus itself: the source current enters and leaves the earthing system at
the same bus, no current flows through the network, and both reduction factors
are undefined there — `res_all_impedances` emits `null` for them. The earthing
impedance is a property of the network at the fault bus and is defined there
as well.

Seven of the ten fault locations add `EARTH_GROUP_SPREAD` to their
diagnostics: the electrode currents of the stations point in clearly
different directions, so the earth current $I_E$ and the reduction factor on
a current basis describe the dominant direction only
([Reduction factor on a current basis](../../explanation/reduction-factors/#reduction-factor-on-a-current-basis)).

In [ ]:
diagnostics = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", gi.ModelAssumptionWarning)  # kept in result.diagnostics
    for i in range(1, N_BUS + 1):
        result = gi.run_fault(net, fault=f"f{i:02d}")
        diagnostics[result.fault] = sorted({d.code for d in result.diagnostics})
print("diagnostics:", {fault: codes for fault, codes in diagnostics.items() if codes})

impedance_df = net.res_all_impedances()
impedance_df.filter(pl.col("frequency_Hz") == 50.0)

## 4. Slice and pivot

`res_all_impedances()` returns one row per (fault, frequency). To plot the
sweep we pivot it to one row per fault bus and one column per frequency.

In [ ]:
def wide(column):
    return impedance_df.pivot(values=column, index="fault_bus", on="frequency_Hz").sort("fault_bus")

ze_wide = wide("grounding_impedance_Ohm")
r_epr_wide = wide("reduction_factor")
r_wide = wide("reduction_factor_current")

print("--- |Z_E| per fault bus ---")
print(ze_wide)
print("\n--- EPR ratio per fault bus ---")
print(r_epr_wide)
print("\n--- EN 50522 reduction factor per fault bus ---")
print(r_wide)

## 5. A second way to $Z_E$

$Z_E$ is the driving-point impedance of the earthing network at the fault bus.
`gi.bus_response` computes, from two solves, the impedance $Z_\text{net}$ that
the rest of the network offers at a bus; in parallel with the bus's own
electrode (1 Ω) it must give $Z_E$.

In [ ]:
response = gi.bus_response(net, fault="f05", bus="bus05")
z_net = response.z_network[50.0]
z_e_second_way = 1.0 / (1.0 / 1.0 + 1.0 / z_net)
z_e_sweep = impedance_df.filter(
    (pl.col("fault_name") == "f05") & (pl.col("frequency_Hz") == 50.0)
)["grounding_impedance_Ohm"][0]
print(f"|Z_E| at bus05, 50 Hz: sweep {z_e_sweep:.6f} ohm, bus_response {abs(z_e_second_way):.6f} ohm")

## 6. Visualisation

Grouped bar plots: one bar per frequency at each fault bus.

In [ ]:
fault_buses = ze_wide["fault_bus"].to_list()
x = np.arange(len(fault_buses))
width = 0.16

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
panels = [
    (axes[0], ze_wide, "|Z_E| / Ohm", "Earthing impedance at the fault bus"),
    (axes[1], r_epr_wide, "EPR ratio", "EPR ratio (reduction factor of the cable)"),
    (axes[2], r_wide, "r", "Reduction factor of EN 50522, |I_E| / |3I_0|"),
]
for ax, frame, ylabel, title in panels:
    for k, f in enumerate(FREQS):
        values = [np.nan if v is None else v for v in frame[str(f)].to_list()]
        ax.bar(x + (k - (len(FREQS) - 1) / 2) * width, values, width=width, label=f"{int(f)} Hz")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
axes[0].legend(ncol=len(FREQS), loc="upper center", fontsize=9)
axes[2].set_xlabel("Fault bus (1 kA injected at bus01)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(fault_buses, rotation=30, ha="right")
fig.tight_layout()
plt.show()

## 7. Profile interpretation

* **$|Z_E|$ is symmetric about the middle of the line.** It is the impedance
  of the earthing network seen from the fault bus — all electrodes in
  parallel through the shields — and does not depend on where the source is.
  The two end stations have bonded neighbours on one side only and show the
  highest value; inside the line it is nearly constant.
* **The EPR ratio is the same at every fault location** at a given frequency:
  every section is the same cable, so it is $|1 - Z_m/Z_s|$ of the cable,
  0.384615 at 50 Hz. It falls with frequency, because the shield reactance
  grows while its resistance stays — to below 0.05 at 550 Hz.
* **The reduction factor of EN 50522 depends on the position.** Close to
  the infeed the shield back to it is short and carries almost all of the
  return current; the farther away the fault, the longer that path and the
  more current takes the soil. At 50 Hz it grows from 0.079 at `bus02` to
  0.389 at `bus10`; at the harmonics, where the shield reactance dominates,
  it levels off after a few stations close to the EPR ratio of that
  frequency. It is the factor that responds to the electrodes and the
  position; the EPR ratio describes the cable.

The wide DataFrames can be exported to CSV or Excel directly via
`polars.DataFrame.write_csv` / `write_excel` for downstream reporting.
[Reduction factors and earthing impedance](../../explanation/reduction-factors/)
explains the two factors.